**READ THE BRONZE DATA AND CREATE DATA FRAME**


In [1]:
# Customer 360 Analytics Platform (Microsoft Fabric)

## Overview
This project demonstrates an end-to-end Customer 360 analytics solution built using Microsoft Fabric.
It integrates customer, order, payment, support, and web activity data into a unified analytics layer
to support marketing, customer service, and leadership decision-making.

## Architecture
- Azure Data Lake Storage Gen2 (source)
- Microsoft Fabric Lakehouse
- Medallion Architecture (Bronze / Silver / Gold)
- PySpark for transformation
- Power BI for reporting

## Data Flow
CSV Sources → Bronze (Parquet + Delta)
→ Silver (Cleaned & Normalized)
→ Gold (Star Schema + Customer 360 Summary)
→ Power BI Semantic Model & Report

## Key KPIs
- Total Active Customers
- Average Order Value (AOV)
- Repeat Customer %
- Open Support Ticket Count
- Churn Risk Indicator
- Payment Method Split
- Device Preference Trends

## Tools Used
- Microsoft Fabric
- PySpark
- Delta Lake
- Power BI

## Notes
This project follows an instructional walkthrough and is enhanced with best-practice
adjustments for real-world scalability, governance, and reporting performance.


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 3, Finished, Available, Finished)

****


**READ BRONZE DATA**

In [3]:
display(customers_raw.limit(5))


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 469a51fc-1253-4626-9cfd-2cb7ecb3b1df)

In [8]:
display(orders_raw.limit(5))


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 10, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a26f392d-6ff7-4b8a-8c84-b76ae2e255c5)

In [9]:
display(payments_raw.limit(5))


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 11, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 79a24c5c-5510-4d46-b0f4-8697b1f071bf)

In [10]:
display(support_raw.limit(5))


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 53de0824-ab54-4a89-99ab-e28c8b6ad46e)

In [11]:
display(web_raw.limit(5))


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, c595f2b6-6fe6-47c2-9131-75c8ea7decab)

**CREATE BRONZE DELTA
 TABLES**

In [12]:
# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")


StatementMeta(, 5a49f4a5-cad4-40ea-927b-7544ffb9c724, 14, Finished, Available, Finished)

**TRANSFORM DATE AND CREATE SILVER TABLES**

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

# -----------------------------
# SILVER LAYER – CLEAN & NORMALIZE
# (Reads your existing tables: customers, orders, payments, support, web)
# Writes silver_* tables
# -----------------------------

# Clean customers
customers = spark.table("customers")

customers_clean = (
    customers
    .withColumn("email", lower(trim(coalesce(col("EMAIL"), col("email")))))
    .withColumn("name", initcap(trim(coalesce(col("NAME"), col("name")))))
    .withColumn(
        "gender",
        when(lower(coalesce(col("GENDER"), col("gender"))).isin("f", "female"), "Female")
        .when(lower(coalesce(col("GENDER"), col("gender"))).isin("m", "male"), "Male")
        .otherwise("Other")
    )
    .withColumn("dob", to_date(regexp_replace(coalesce(col("DOB"), col("dob")), "/", "-")))
    .withColumn("location", initcap(coalesce(col("LOCATION"), col("location"))))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)

customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")


# Clean orders
orders = spark.table("orders")

orders_clean = (
    orders
    .withColumn(
        "order_date",
        when(col("order_date").rlike(r"^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
        .when(col("order_date").rlike(r"^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
        .when(col("order_date").rlike(r"^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
        .otherwise(to_date(col("order_date"), "yyyy-MM-dd"))
    )
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)

orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")


# Clean payments
payments = spark.table("payments")

payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)

payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")


# Clean support
support = spark.table("support")

support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")


# Clean web
web = spark.table("web")

web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)

web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")


StatementMeta(, e4d6183d-1360-4add-a1f4-526cec0c5ac8, 3, Finished, Available, Finished)

**GOLD TABLES-AGGREGATE TABLES
**

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# -----------------------------
# GOLD LAYER – BEST PRACTICE (STAR + CUSTOMER SUMMARY)
# Source: silver_* tables
# Target: gold_* tables
# -----------------------------

cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

# -----------------------------
# 1) GOLD DIMENSION: Customer
# -----------------------------
gold_dim_customer = (
    cust
    .select(
        col("customer_id"),
        col("name"),
        col("email"),
        col("gender"),
        col("dob"),
        col("location")
    )
    .dropDuplicates(["customer_id"])
)

gold_dim_customer.write.format("delta").mode("overwrite").saveAsTable("gold_dim_customer")


# -----------------------------
# 2) GOLD FACT: Orders
# -----------------------------
gold_fact_orders = (
    orders
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_date"),
        col("amount").alias("order_amount"),
        col("status").alias("order_status")
    )
    .dropDuplicates(["order_id"])
)

gold_fact_orders.write.format("delta").mode("overwrite").saveAsTable("gold_fact_orders")


# -----------------------------
# 3) GOLD FACT: Payments
# -----------------------------
# NOTE: If payments has a payment_id column, include it. If not, we create a deterministic key.
payment_cols = payments.columns
has_payment_id = "payment_id" in payment_cols

gold_fact_payments = (
    payments
    .withColumn(
        "payment_id",
        col("payment_id") if has_payment_id else sha2(concat_ws("||", col("customer_id").cast("string"),
                                                               col("payment_date").cast("string"),
                                                               col("amount").cast("string"),
                                                               col("payment_method").cast("string"),
                                                               col("payment_status").cast("string")), 256)
    )
    .select(
        col("payment_id"),
        col("customer_id"),
        col("payment_date"),
        col("amount").alias("payment_amount"),
        col("payment_method"),
        col("payment_status")
    )
    .dropDuplicates(["payment_id"])
)

gold_fact_payments.write.format("delta").mode("overwrite").saveAsTable("gold_fact_payments")


# -----------------------------
# 4) GOLD FACT: Support Tickets
# -----------------------------
gold_fact_support = (
    support
    .select(
        col("ticket_id"),
        col("customer_id"),
        col("ticket_date"),
        col("issue_type"),
        col("resolution_status")
    )
    .dropDuplicates(["ticket_id"])
)

gold_fact_support.write.format("delta").mode("overwrite").saveAsTable("gold_fact_support")


# -----------------------------
# 5) GOLD FACT: Web Sessions / Activity
# -----------------------------
# NOTE: If web has session_id, use it; otherwise create a deterministic key.
web_cols = web.columns
has_session_id = "session_id" in web_cols

gold_fact_web = (
    web
    .withColumn(
        "session_id",
        col("session_id") if has_session_id else sha2(concat_ws("||", col("customer_id").cast("string"),
                                                               col("session_time").cast("string"),
                                                               col("page_viewed").cast("string"),
                                                               col("device_type").cast("string")), 256)
    )
    .select(
        col("session_id"),
        col("customer_id"),
        col("session_time"),
        col("page_viewed"),
        col("device_type")
    )
    .dropDuplicates(["session_id"])
)

gold_fact_web.write.format("delta").mode("overwrite").saveAsTable("gold_fact_web")


# -----------------------------
# 6) GOLD AGGREGATE: Customer 360 Summary (Best for KPI cards & segmentation)
# -----------------------------
# Business-friendly rollup per customer (prevents row explosion in BI)

# Orders summary
orders_by_customer = (
    gold_fact_orders
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("order_count"),
        sum("order_amount").alias("total_order_amount"),
        avg("order_amount").alias("avg_order_value"),
        max("order_date").alias("last_order_date")
    )
)

# Payments summary (mode + split helper fields)
payments_by_customer = (
    gold_fact_payments
    .groupBy("customer_id")
    .agg(
        count("*").alias("payment_count"),
        sum("payment_amount").alias("total_paid_amount"),
        max("payment_date").alias("last_payment_date")
    )
)

# Support summary (open tickets = unresolved)
support_by_customer = (
    gold_fact_support
    .groupBy("customer_id")
    .agg(
        countDistinct("ticket_id").alias("ticket_count"),
        sum(when(lower(col("resolution_status")).isin("open", "pending", "unresolved"), 1).otherwise(0)).alias("open_ticket_count"),
        max("ticket_date").alias("last_ticket_date")
    )
)

# Web summary (device preference)
device_pref = (
    gold_fact_web
    .groupBy("customer_id")
    .agg(
        sum(when(lower(col("device_type")).like("%mobile%"), 1).otherwise(0)).alias("mobile_sessions"),
        sum(when(lower(col("device_type")).like("%web%") | lower(col("device_type")).like("%desktop%"), 1).otherwise(0)).alias("web_sessions"),
        countDistinct("session_id").alias("total_sessions"),
        max("session_time").alias("last_session_date")
    )
    .withColumn(
        "device_preference",
        when(col("mobile_sessions") > col("web_sessions"), "Mobile")
        .when(col("web_sessions") > col("mobile_sessions"), "Web")
        .otherwise("Mixed")
    )
)

# Assemble Customer 360 Summary
gold_customer360_summary = (
    gold_dim_customer.alias("d")
    .join(orders_by_customer.alias("o"), "customer_id", "left")
    .join(payments_by_customer.alias("p"), "customer_id", "left")
    .join(support_by_customer.alias("s"), "customer_id", "left")
    .join(device_pref.alias("w"), "customer_id", "left")
    .fillna({
        "order_count": 0,
        "total_order_amount": 0.0,
        "avg_order_value": 0.0,
        "payment_count": 0,
        "total_paid_amount": 0.0,
        "ticket_count": 0,
        "open_ticket_count": 0,
        "mobile_sessions": 0,
        "web_sessions": 0,
        "total_sessions": 0
    })
    # Repeat Customer flag (% repeat customers KPI will use this)
    .withColumn("is_repeat_customer", when(col("order_count") >= 2, lit(1)).otherwise(lit(0)))
    # Active Customer flag (at least one order)
    .withColumn("is_active_customer", when(col("order_count") >= 1, lit(1)).otherwise(lit(0)))
    # Churn Risk (your definition: past tickets AND no recent orders)
    # Best-practice parameterization: set threshold (e.g., 60 days) as a variable
)

# Churn threshold (best practice: make this easy to tune)
CHURN_DAYS = 60
gold_customer360_summary = (
    gold_customer360_summary
    .withColumn("days_since_last_order", datediff(current_date(), col("last_order_date")))
    .withColumn("has_past_tickets", when(col("ticket_count") > 0, lit(1)).otherwise(lit(0)))
    .withColumn(
        "is_churn_risk",
        when((col("has_past_tickets") == 1) & (col("last_order_date").isNull() | (col("days_since_last_order") > CHURN_DAYS)), lit(1)).otherwise(lit(0))
    )
)

gold_customer360_summary.write.format("delta").mode("overwrite").saveAsTable("gold_customer360_summary")


StatementMeta(, e4d6183d-1360-4add-a1f4-526cec0c5ac8, 4, Finished, Available, Finished)

**Gold Layer-Validate Outputs
**

In [3]:
# List all tables in the Lakehouse and show only GOLD tables
tables = spark.sql("SHOW TABLES").select("tableName")
gold_tables = tables.filter(col("tableName").startswith("gold_")).orderBy("tableName")

display(gold_tables)


StatementMeta(, e4d6183d-1360-4add-a1f4-526cec0c5ac8, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ffdf640e-9727-4d50-88dc-bded19d074cd)

In [4]:
# Preview row counts + sample rows for each Gold table
gold_table_names = [r["tableName"] for r in gold_tables.collect()]

for t in gold_table_names:
    print(f"\n===== {t} =====")
    print("Row count:", spark.table(t).count())
    display(spark.table(t).limit(5))


StatementMeta(, e4d6183d-1360-4add-a1f4-526cec0c5ac8, 6, Finished, Available, Finished)


===== gold_customer360_summary =====
Row count: 15


SynapseWidget(Synapse.DataFrame, 9c020881-aa0e-4feb-a23f-6d07c68a7f84)


===== gold_dim_customer =====
Row count: 15


SynapseWidget(Synapse.DataFrame, 4105a708-9e81-41f8-938a-7e0c6003835e)

...

SynapseWidget(Synapse.DataFrame, 452030db-e9e9-41f6-938c-3898ae85cfeb)


===== gold_fact_support =====
Row count: 8


SynapseWidget(Synapse.DataFrame, b4a894d2-0492-430d-a69e-cba7ab606f7b)


===== gold_fact_web =====
Row count: 6


SynapseWidget(Synapse.DataFrame, ee8014ec-7d0d-4370-acce-d5bb31a02a68)

In [5]:
# Schema check for the main Customer 360 summary table
spark.table("gold_customer360_summary").printSchema()


StatementMeta(, e4d6183d-1360-4add-a1f4-526cec0c5ac8, 7, Finished, Available, Finished)

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- location: string (nullable = true)
 |-- order_count: long (nullable = true)
 |-- total_order_amount: double (nullable = true)
 |-- avg_order_value: double (nullable = true)
 |-- last_order_date: date (nullable = true)
 |-- payment_count: long (nullable = true)
 |-- total_paid_amount: double (nullable = true)
 |-- last_payment_date: date (nullable = true)
 |-- ticket_count: long (nullable = true)
 |-- open_ticket_count: long (nullable = true)
 |-- last_ticket_date: date (nullable = true)
 |-- mobile_sessions: long (nullable = true)
 |-- web_sessions: long (nullable = true)
 |-- total_sessions: long (nullable = true)
 |-- last_session_date: date (nullable = true)
 |-- device_preference: string (nullable = true)
 |-- is_repeat_customer: integer (nullable = true)
 |-- is_active_customer: integer 